In [ ]:
# ============================================================
# PT -> PA Cross-Physics Mapping
# Latent Neural Operator (LNO2D)
#
# Input:
#     PT [B, 1, 501, 200]
#
# Output:
#     PA [B, 1, 501, 200]
#
# Architecture:
#
# PT
# [501 x 200]
#
#       |
#       v
#
# PT Encoder
#
#       |
#       v
#
# Latent Field
# [~63 x 25]
#
#       |
#       v
#
# Latent Neural Operator
# Fourier operator blocks
#
#       |
#       v
#
# PA Decoder
#
#       |
#       v
#
# PA
# [501 x 200]
#
# ============================================================


# ============================================================
# Imports
# ============================================================

import os
import csv
import glob
import random
import time
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# Configuration
# ============================================================

DATA_ROOT = "Training dataset"


RUN_TIME = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUT_DIR = os.path.join(
    "latent_neural_operator_results",
    f"run_{RUN_TIME}"
)


os.makedirs(
    OUT_DIR,
    exist_ok=True
)


print("Results will be saved to:")
print(OUT_DIR)


# ============================================================
# Dataset split
# ============================================================

TRAIN_RATIO = 0.8

VAL_RATIO = 0.1

TEST_RATIO = 0.1

SEED = 42


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Using device:", DEVICE)


# ============================================================
# Data dimensions
# ============================================================

NT = 501

NX = 200


# ============================================================
# Training parameters
# ============================================================

EPOCHS = 1000

BATCH_SIZE = 4

LR = 1e-3

WEIGHT_DECAY = 1e-5


# ============================================================
# LNO parameters
# ============================================================

IN_CHANNELS = 1

OUT_CHANNELS = 1


# ------------------------------------------------------------
# Latent feature width
# ------------------------------------------------------------

LATENT_WIDTH = 128


# ------------------------------------------------------------
# Number of latent operator layers
# ------------------------------------------------------------

N_OPERATOR_LAYERS = 4


# ------------------------------------------------------------
# Fourier modes used inside latent space
#
# Latent resolution is approximately:
#
# 501 x 200
#
# ->
#
# 251 x 100
#
# ->
#
# 126 x 50
#
# ->
#
# 63 x 25
#
# Therefore:
#
# T modes <= 31
# X rFFT modes <= 13
#
# ------------------------------------------------------------

LATENT_MODES_T = 24

LATENT_MODES_X = 12


# ============================================================
# Random seed
# ============================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# Find all .mat files
# ============================================================

all_files = glob.glob(

    os.path.join(
        DATA_ROOT,
        "**",
        "*.mat"
    ),

    recursive=True
)


all_files = sorted(
    all_files
)


print(
    "Total .mat samples found:",
    len(all_files)
)


assert len(all_files) > 0, \
    f"No .mat files found under: {DATA_ROOT}"


if len(all_files) != 1000:

    print(
        f"Warning: expected 1000 samples, "
        f"but found {len(all_files)}"
    )


# ============================================================
# Random dataset split
# ============================================================

random.shuffle(
    all_files
)


n_total = len(
    all_files
)


n_train = int(
    n_total * TRAIN_RATIO
)


n_val = int(
    n_total * VAL_RATIO
)


train_files = all_files[
    :n_train
]


val_files = all_files[
    n_train:
    n_train + n_val
]


test_files = all_files[
    n_train + n_val:
]


print("\nDataset split:")


print(
    "Train samples:",
    len(train_files)
)


print(
    "Val samples  :",
    len(val_files)
)


print(
    "Test samples :",
    len(test_files)
)


# ============================================================
# Compute normalization statistics
#
# IMPORTANT:
#
# Only training data are used.
# ============================================================

def compute_statistics(
    file_list
):


    pt_sum = 0.0

    pt_sq_sum = 0.0


    pa_sum = 0.0

    pa_sq_sum = 0.0


    n_elements = 0


    print(
        "\nCalculating normalization statistics..."
    )


    for mat_path in tqdm(

        file_list,

        desc="Statistics",

        ncols=120
    ):


        with h5py.File(
            mat_path,
            "r"
        ) as f:


            PT = np.asarray(
                f["PT"],
                dtype=np.float64
            )


            PA = np.asarray(
                f["PA"],
                dtype=np.float64
            )


        # ====================================================
        # Shape
        # ====================================================

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"


        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        pt_sum += PT.sum()


        pt_sq_sum += np.square(
            PT
        ).sum()


        pa_sum += PA.sum()


        pa_sq_sum += np.square(
            PA
        ).sum()


        n_elements += PT.size


    # ========================================================
    # Mean
    # ========================================================

    pt_mean = (
        pt_sum
        /
        n_elements
    )


    pa_mean = (
        pa_sum
        /
        n_elements
    )


    # ========================================================
    # Variance
    # ========================================================

    pt_var = (
        pt_sq_sum
        /
        n_elements
        -
        pt_mean ** 2
    )


    pa_var = (
        pa_sq_sum
        /
        n_elements
        -
        pa_mean ** 2
    )


    pt_var = max(
        pt_var,
        0.0
    )


    pa_var = max(
        pa_var,
        0.0
    )


    # ========================================================
    # Standard deviation
    # ========================================================

    pt_std = (
        np.sqrt(
            pt_var
        )
        +
        1e-8
    )


    pa_std = (
        np.sqrt(
            pa_var
        )
        +
        1e-8
    )


    return (
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    )


# ============================================================
# Calculate normalization
# ============================================================

(
    pt_mean,
    pt_std,
    pa_mean,
    pa_std

) = compute_statistics(
    train_files
)


print(
    "\nNormalization statistics:"
)


print(
    f"PT mean = {pt_mean:.6e}"
)


print(
    f"PT std  = {pt_std:.6e}"
)


print(
    f"PA mean = {pa_mean:.6e}"
)


print(
    f"PA std  = {pa_std:.6e}"
)


# ============================================================
# Save normalization parameters
# ============================================================

np.savez(

    os.path.join(
        OUT_DIR,
        "normalization_parameters.npz"
    ),

    pt_mean=pt_mean,

    pt_std=pt_std,

    pa_mean=pa_mean,

    pa_std=pa_std
)


# ============================================================
# Dataset
# ============================================================

class MatHeatAcousticDataset(Dataset):


    def __init__(
        self,
        file_list,
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    ):


        self.file_list = file_list


        self.pt_mean = pt_mean

        self.pt_std = pt_std


        self.pa_mean = pa_mean

        self.pa_std = pa_std


    # ========================================================
    # Dataset length
    # ========================================================

    def __len__(
        self
    ):


        return len(
            self.file_list
        )


    # ========================================================
    # Sample
    # ========================================================
    import time

    def __getitem__(self, idx):
    
        mat_path = self.file_list[idx]
    
        max_retries = 20
    
        for attempt in range(max_retries):
    
            try:
    
                with h5py.File(mat_path, "r") as f:
    
                    PT = np.asarray(
                        f["PT"],
                        dtype=np.float32
                    )
    
                    PA = np.asarray(
                        f["PA"],
                        dtype=np.float32
                    )
    
                break
    
            except OSError as e:
    
                print(
                    f"\nTemporary file access error:\n"
                    f"{mat_path}\n"
                    f"Attempt {attempt + 1}/{max_retries}\n"
                    f"{e}"
                )
    
                if attempt == max_retries - 1:
                    raise
    
                time.sleep(10)
    
        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"
    
        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"
    
        PT = (
            PT - self.pt_mean
        ) / self.pt_std
    
        PA = (
            PA - self.pa_mean
        ) / self.pa_std
    
        PT = torch.tensor(
            PT,
            dtype=torch.float32
        ).unsqueeze(0)
    
        PA = torch.tensor(
            PA,
            dtype=torch.float32
        ).unsqueeze(0)
    
        return PT, PA


    # ========================================================
    # Denormalize
    # ========================================================

    def denormalize_pa(
        self,
        x
    ):


        return (
            x
            *
            self.pa_std
            +
            self.pa_mean
        )


    def denormalize_pt(
        self,
        x
    ):


        return (
            x
            *
            self.pt_std
            +
            self.pt_mean
        )


# ============================================================
# Datasets
# ============================================================

train_dataset = MatHeatAcousticDataset(

    train_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


val_dataset = MatHeatAcousticDataset(

    val_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


test_dataset = MatHeatAcousticDataset(

    test_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


# ============================================================
# DataLoaders
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


# ============================================================
# Dataset check
# ============================================================

PT_batch, PA_batch = next(
    iter(
        train_loader
    )
)


print(
    "\nBatch check:"
)


print(
    "PT batch shape:",
    PT_batch.shape
)


print(
    "PA batch shape:",
    PA_batch.shape
)


# ============================================================
# Basic convolution block
# ============================================================

class ConvBlock(nn.Module):


    def __init__(
        self,
        in_channels,
        out_channels
    ):


        super().__init__()


        groups = min(
            8,
            out_channels
        )


        while (
            out_channels % groups != 0
            and
            groups > 1
        ):

            groups -= 1


        self.block = nn.Sequential(

            nn.Conv2d(

                in_channels,

                out_channels,

                kernel_size=3,

                padding=1,

                bias=False
            ),

            nn.GroupNorm(
                groups,
                out_channels
            ),

            nn.GELU(),


            nn.Conv2d(

                out_channels,

                out_channels,

                kernel_size=3,

                padding=1,

                bias=False
            ),

            nn.GroupNorm(
                groups,
                out_channels
            ),

            nn.GELU()
        )


        # ====================================================
        # Residual connection
        # ====================================================

        if in_channels != out_channels:


            self.shortcut = nn.Conv2d(

                in_channels,

                out_channels,

                kernel_size=1
            )


        else:


            self.shortcut = nn.Identity()


    def forward(
        self,
        x
    ):


        residual = self.shortcut(
            x
        )


        x = self.block(
            x
        )


        return (
            x
            +
            residual
        )


# ============================================================
# PT Encoder
#
# Physical PT field
#
# ->
#
# compact latent representation
# ============================================================

class PTEncoder(nn.Module):


    def __init__(
        self,
        latent_width=128
    ):


        super().__init__()


        # ====================================================
        # Level 1
        #
        # 501 x 200
        # ====================================================

        self.enc1 = ConvBlock(

            3,

            32
        )


        # ====================================================
        # Level 2
        #
        # ~251 x 100
        # ====================================================

        self.enc2 = ConvBlock(

            32,

            64
        )


        # ====================================================
        # Level 3
        #
        # ~126 x 50
        # ====================================================

        self.enc3 = ConvBlock(

            64,

            96
        )


        # ====================================================
        # Latent
        #
        # ~63 x 25
        # ====================================================

        self.enc4 = ConvBlock(

            96,

            latent_width
        )


    def forward(
        self,
        x
    ):


        # ====================================================
        # 501 x 200
        # ====================================================

        x = self.enc1(
            x
        )


        # ====================================================
        # 251 x 100
        # ====================================================

        x = F.avg_pool2d(

            x,

            kernel_size=2,

            stride=2,

            ceil_mode=True
        )


        x = self.enc2(
            x
        )


        # ====================================================
        # 126 x 50
        # ====================================================

        x = F.avg_pool2d(

            x,

            kernel_size=2,

            stride=2,

            ceil_mode=True
        )


        x = self.enc3(
            x
        )


        # ====================================================
        # 63 x 25
        # ====================================================

        x = F.avg_pool2d(

            x,

            kernel_size=2,

            stride=2,

            ceil_mode=True
        )


        x = self.enc4(
            x
        )


        return x


# ============================================================
# Latent Spectral Convolution
#
# The operator is learned ONLY in latent space.
#
# This greatly reduces the cost relative to applying FNO
# directly on 501 x 200.
# ============================================================

class LatentSpectralConv2d(nn.Module):


    def __init__(
        self,
        in_channels,
        out_channels,
        modes_t,
        modes_x
    ):


        super().__init__()


        self.in_channels = in_channels

        self.out_channels = out_channels


        self.modes_t = modes_t

        self.modes_x = modes_x


        scale = (
            1.0
            /
            (
                in_channels
                *
                out_channels
            )
        )


        # ====================================================
        # Positive temporal frequencies
        # ====================================================

        self.weights_pos = nn.Parameter(

            scale
            *
            torch.randn(

                in_channels,

                out_channels,

                modes_t,

                modes_x,

                dtype=torch.cfloat
            )
        )


        # ====================================================
        # Negative temporal frequencies
        # ====================================================

        self.weights_neg = nn.Parameter(

            scale
            *
            torch.randn(

                in_channels,

                out_channels,

                modes_t,

                modes_x,

                dtype=torch.cfloat
            )
        )


    # ========================================================
    # Complex multiplication
    # ========================================================

    def compl_mul2d(
        self,
        x,
        weights
    ):


        return torch.einsum(

            "bixy,ioxy->boxy",

            x,

            weights
        )


    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        x
    ):


        B, C, T, X = x.shape


        # ====================================================
        # FFT
        # ====================================================

        x_ft = torch.fft.rfftn(

            x,

            dim=(-2, -1)
        )


        # ====================================================
        # Output Fourier coefficients
        # ====================================================

        out_ft = torch.zeros(

            B,

            self.out_channels,

            T,

            X // 2 + 1,

            dtype=torch.cfloat,

            device=x.device
        )


        # ====================================================
        # Safe modes
        # ====================================================

        mt = min(

            self.modes_t,

            T // 2
        )


        mx = min(

            self.modes_x,

            X // 2 + 1
        )


        # ====================================================
        # Positive temporal modes
        # ====================================================

        if mt > 0 and mx > 0:


            out_ft[
                :,
                :,
                :mt,
                :mx
            ] = self.compl_mul2d(

                x_ft[
                    :,
                    :,
                    :mt,
                    :mx
                ],

                self.weights_pos[
                    :,
                    :,
                    :mt,
                    :mx
                ]
            )


            # =================================================
            # Negative temporal modes
            # =================================================

            out_ft[
                :,
                :,
                -mt:,
                :mx
            ] = self.compl_mul2d(

                x_ft[
                    :,
                    :,
                    -mt:,
                    :mx
                ],

                self.weights_neg[
                    :,
                    :,
                    :mt,
                    :mx
                ]
            )


        # ====================================================
        # Inverse FFT
        # ====================================================

        x = torch.fft.irfftn(

            out_ft,

            s=(T, X),

            dim=(-2, -1)
        )


        return x


# ============================================================
# Latent Neural Operator Block
#
# Spectral operator
#
# +
#
# local operator
#
# +
#
# residual connection
# ============================================================

class LatentOperatorBlock(nn.Module):


    def __init__(
        self,
        channels,
        modes_t,
        modes_x
    ):


        super().__init__()


        # ====================================================
        # Global spectral operator
        # ====================================================

        self.spectral = LatentSpectralConv2d(

            channels,

            channels,

            modes_t,

            modes_x
        )


        # ====================================================
        # Local operator
        # ====================================================

        self.local = nn.Conv2d(

            channels,

            channels,

            kernel_size=1
        )


        # ====================================================
        # Local spatial refinement
        # ====================================================

        self.refine = nn.Conv2d(

            channels,

            channels,

            kernel_size=3,

            padding=1
        )


        groups = min(
            8,
            channels
        )


        while (
            channels % groups != 0
            and
            groups > 1
        ):

            groups -= 1


        self.norm = nn.GroupNorm(

            groups,

            channels
        )


    def forward(
        self,
        x
    ):


        residual = x


        # ====================================================
        # Spectral branch
        # ====================================================

        x1 = self.spectral(
            x
        )


        # ====================================================
        # Local branch
        # ====================================================

        x2 = self.local(
            x
        )


        # ====================================================
        # Combine
        # ====================================================

        x = (
            x1
            +
            x2
        )


        x = self.norm(
            x
        )


        x = F.gelu(
            x
        )


        # ====================================================
        # Local refinement
        # ====================================================

        x = (
            x
            +
            self.refine(
                x
            )
        )


        # ====================================================
        # Residual
        # ====================================================

        x = (
            x
            +
            residual
        )


        x = F.gelu(
            x
        )


        return x


# ============================================================
# Latent Neural Operator
#
# z_PT
#
# ->
#
# z_PA
# ============================================================

class LatentOperator(nn.Module):


    def __init__(
        self,
        channels,
        modes_t,
        modes_x,
        n_layers
    ):


        super().__init__()


        self.layers = nn.ModuleList(

            [

                LatentOperatorBlock(

                    channels=channels,

                    modes_t=modes_t,

                    modes_x=modes_x
                )

                for _ in range(
                    n_layers
                )
            ]
        )


        # ====================================================
        # Additional latent mapping
        #
        # Important for cross-physics:
        #
        # PT latent space
        #
        # ->
        #
        # PA latent representation
        # ====================================================

        self.cross_physics_projection = nn.Sequential(

            nn.Conv2d(

                channels,

                channels,

                kernel_size=1
            ),

            nn.GELU(),

            nn.Conv2d(

                channels,

                channels,

                kernel_size=1
            )
        )


    def forward(
        self,
        x
    ):


        # ====================================================
        # Neural operator layers
        # ====================================================

        for layer in self.layers:


            x = layer(
                x
            )


        # ====================================================
        # Explicit PT-latent -> PA-latent mapping
        # ====================================================

        residual = x


        x = self.cross_physics_projection(
            x
        )


        x = (
            x
            +
            residual
        )


        return x


# ============================================================
# PA Decoder
#
# Latent PA representation
#
# ->
#
# physical PA field
# ============================================================

class PADecoder(nn.Module):


    def __init__(
        self,
        latent_width=128,
        out_channels=1
    ):


        super().__init__()


        # ====================================================
        # 63 x 25
        # ====================================================

        self.dec3 = ConvBlock(

            latent_width,

            96
        )


        # ====================================================
        # 126 x 50
        # ====================================================

        self.dec2 = ConvBlock(

            96,

            64
        )


        # ====================================================
        # 251 x 100
        # ====================================================

        self.dec1 = ConvBlock(

            64,

            32
        )


        # ====================================================
        # 501 x 200
        # ====================================================

        self.final_refine = nn.Sequential(

            nn.Conv2d(

                32,

                32,

                kernel_size=3,

                padding=1
            ),

            nn.GELU(),


            nn.Conv2d(

                32,

                16,

                kernel_size=3,

                padding=1
            ),

            nn.GELU()
        )


        self.output = nn.Conv2d(

            16,

            out_channels,

            kernel_size=1
        )


    def forward(
        self,
        x,
        output_size
    ):


        target_t = output_size[
            0
        ]


        target_x = output_size[
            1
        ]


        # ====================================================
        # Encoder resolutions for original 501 x 200:
        #
        # latent:
        # 63 x 25
        #
        # ->
        #
        # 126 x 50
        # ====================================================

        size2 = (

            int(
                np.ceil(
                    target_t / 4
                )
            ),

            int(
                np.ceil(
                    target_x / 4
                )
            )
        )


        x = F.interpolate(

            x,

            size=size2,

            mode="bilinear",

            align_corners=False
        )


        x = self.dec3(
            x
        )


        # ====================================================
        # 251 x 100
        # ====================================================

        size1 = (

            int(
                np.ceil(
                    target_t / 2
                )
            ),

            int(
                np.ceil(
                    target_x / 2
                )
            )
        )


        x = F.interpolate(

            x,

            size=size1,

            mode="bilinear",

            align_corners=False
        )


        x = self.dec2(
            x
        )


        # ====================================================
        # 501 x 200
        # ====================================================

        x = F.interpolate(

            x,

            size=(
                target_t,
                target_x
            ),

            mode="bilinear",

            align_corners=False
        )


        x = self.dec1(
            x
        )


        # ====================================================
        # PA local reconstruction
        # ====================================================

        x = self.final_refine(
            x
        )


        x = self.output(
            x
        )


        return x


# ============================================================
# Complete Latent Neural Operator
#
# PT
#
# ->
#
# E_PT
#
# ->
#
# z_PT
#
# ->
#
# G_theta
#
# ->
#
# z_PA
#
# ->
#
# D_PA
#
# ->
#
# PA
# ============================================================

class LatentNeuralOperator2D(nn.Module):


    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        latent_width=128,
        modes_t=24,
        modes_x=12,
        n_operator_layers=4
    ):


        super().__init__()


        # ====================================================
        # PT Encoder
        #
        # Input channels:
        #
        # PT
        # +
        # t coordinate
        # +
        # x coordinate
        #
        # = 3 channels
        # ====================================================

        self.encoder = PTEncoder(

            latent_width=latent_width
        )


        # ====================================================
        # Latent operator
        # ====================================================

        self.operator = LatentOperator(

            channels=latent_width,

            modes_t=modes_t,

            modes_x=modes_x,

            n_layers=n_operator_layers
        )


        # ====================================================
        # PA Decoder
        # ====================================================

        self.decoder = PADecoder(

            latent_width=latent_width,

            out_channels=out_channels
        )


    # ========================================================
    # Coordinate grid
    # ========================================================

    def get_grid(
        self,
        shape,
        device
    ):


        B, C, T, X = shape


        t = torch.linspace(

            0.0,

            1.0,

            T,

            device=device
        )


        x = torch.linspace(

            0.0,

            1.0,

            X,

            device=device
        )


        tt, xx = torch.meshgrid(

            t,

            x,

            indexing="ij"
        )


        grid = torch.stack(

            [
                tt,
                xx
            ],

            dim=0
        )


        grid = grid.unsqueeze(
            0
        )


        grid = grid.repeat(

            B,

            1,

            1,

            1
        )


        return grid


    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        pt
    ):


        original_size = pt.shape[
            -2:
        ]


        # ====================================================
        # Coordinates
        # ====================================================

        grid = self.get_grid(

            pt.shape,

            pt.device
        )


        # ====================================================
        # PT + coordinates
        #
        # [B,3,501,200]
        # ====================================================

        x = torch.cat(

            [
                pt,
                grid
            ],

            dim=1
        )


        # ====================================================
        # Encode PT
        #
        # ->
        #
        # latent PT representation
        #
        # approximately:
        #
        # [B,128,63,25]
        # ====================================================

        z_pt = self.encoder(
            x
        )


        # ====================================================
        # Cross-physics latent operator
        #
        # z_PT
        #
        # ->
        #
        # z_PA
        # ====================================================

        z_pa = self.operator(
            z_pt
        )


        # ====================================================
        # Decode PA
        # ====================================================

        pa = self.decoder(

            z_pa,

            output_size=original_size
        )


        return pa


# ============================================================
# Build model
# ============================================================

model = LatentNeuralOperator2D(

    in_channels=IN_CHANNELS,

    out_channels=OUT_CHANNELS,

    latent_width=LATENT_WIDTH,

    modes_t=LATENT_MODES_T,

    modes_x=LATENT_MODES_X,

    n_operator_layers=N_OPERATOR_LAYERS

).to(
    DEVICE
)


# ============================================================
# Model shape check
# ============================================================

print(
    "\nChecking model dimensions..."
)


with torch.no_grad():


    test_input = torch.randn(

        1,

        1,

        NT,

        NX,

        device=DEVICE
    )


    test_output = model(
        test_input
    )


print(
    "Input shape :",
    test_input.shape
)


print(
    "Output shape:",
    test_output.shape
)


assert (

    test_output.shape

    ==

    test_input.shape

), \
    f"Output shape mismatch: {test_output.shape}"


del test_input

del test_output


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# Number of trainable parameters
# ============================================================

n_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(
    f"\nTrainable parameters: {n_params:,}"
)


# ============================================================
# Relative L2
# ============================================================

def relative_l2(
    pred,
    target
):


    numerator = torch.norm(

        pred
        -
        target
    )


    denominator = (

        torch.norm(
            target
        )

        +

        1e-8
    )


    return (
        numerator
        /
        denominator
    )


# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):


    model.train()


    total_mse = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        # ====================================================
        # Forward
        # ====================================================

        pred = model(
            heat
        )


        # ====================================================
        # Same loss as previous models
        # ====================================================

        mse = F.mse_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        loss = (

            mse

            +

            0.1
            *
            rel
        )


        # ====================================================
        # Backpropagation
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0
        )


        optimizer.step()


        total_mse += mse.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):


    model.eval()


    total_mse = 0.0

    total_mae = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        pred = model(
            heat
        )


        mse = F.mse_loss(

            pred,

            acoustic
        )


        mae = F.l1_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        total_mse += mse.item()

        total_mae += mae.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_mae
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Plot loss curves
# ============================================================

def plot_loss(
    log_path
):


    data = np.loadtxt(

        log_path,

        delimiter=",",

        skiprows=1
    )


    if data.ndim == 1:

        data = data[
            None,
            :
        ]


    epoch = data[
        :,
        0
    ]


    train_mse = data[
        :,
        1
    ]


    train_rel = data[
        :,
        2
    ]


    val_mse = data[
        :,
        3
    ]


    val_rel = data[
        :,
        5
    ]


    # ========================================================
    # MSE
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_mse,

        label="Train MSE"
    )


    plt.semilogy(

        epoch,

        val_mse,

        label="Validation MSE"
    )


    plt.xlabel(
        "Epoch"
    )


    plt.ylabel(
        "MSE"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "loss_curve.png"
        ),

        dpi=300
    )


    plt.close()


    # ========================================================
    # Relative L2
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_rel,

        label="Train Rel. L2"
    )


    plt.semilogy(

        epoch,

        val_rel,

        label="Validation Rel. L2"
    )


    plt.xlabel(
        "Epoch"
    )


    plt.ylabel(
        "Relative L2"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "relative_l2_curve.png"
        ),

        dpi=300
    )


    plt.close()


# ============================================================
# Prediction plot
# ============================================================

@torch.no_grad()
def plot_prediction(
    model,
    dataset,
    sample_index=0
):


    model.eval()


    PT, PA = dataset[
        sample_index
    ]


    # ========================================================
    # Predict
    # ========================================================

    PT_gpu = PT.unsqueeze(
        0
    ).to(
        DEVICE
    )


    pred = model(
        PT_gpu
    )


    pred = (

        pred
        .cpu()
        .squeeze(0)
        .squeeze(0)
        .numpy()
    )


    PT = (

        PT
        .squeeze(0)
        .numpy()
    )


    PA = (

        PA
        .squeeze(0)
        .numpy()
    )


    # ========================================================
    # Denormalization
    # ========================================================

    PT_real = dataset.denormalize_pt(
        PT
    )


    PA_real = dataset.denormalize_pa(
        PA
    )


    pred_real = dataset.denormalize_pa(
        pred
    )


    # ========================================================
    # Error
    # ========================================================

    err = (

        pred_real

        -

        PA_real
    )


    # ========================================================
    # Color scale
    # ========================================================

    vmax = np.max(

        np.abs(
            PA_real
        )
    )


    vmax = max(
        vmax,
        1e-12
    )


    evmax = np.max(

        np.abs(
            err
        )
    )


    evmax = max(
        evmax,
        1e-12
    )


    # ========================================================
    # Figure
    # ========================================================

    plt.figure(
        figsize=(12, 3)
    )


    # --------------------------------------------------------
    # PT
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        1
    )


    plt.imshow(

        PT_real,

        aspect="auto",

        cmap="inferno"
    )


    plt.title(
        "Input PT"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # GT PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        2
    )


    plt.imshow(

        PA_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "GT PA"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Predicted PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        3
    )


    plt.imshow(

        pred_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "Pred PA"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Error
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        4
    )


    plt.imshow(

        err,

        aspect="auto",

        cmap="seismic",

        vmin=-evmax,

        vmax=evmax
    )


    plt.title(
        "Error"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "prediction_comparison.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.close()


    # ========================================================
    # Save numerical prediction
    # ========================================================

    np.savez(

        os.path.join(
            OUT_DIR,
            "prediction_result.npz"
        ),

        PT=PT_real,

        PA=PA_real,

        PA_pred=pred_real,

        error=err
    )


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# Learning-rate scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=EPOCHS
)


# ============================================================
# Training log
# ============================================================

log_path = os.path.join(

    OUT_DIR,

    "training_log.csv"
)


with open(

    log_path,

    "w",

    newline=""

) as f:


    writer = csv.writer(
        f
    )


    writer.writerow(

        [
            "epoch",
            "train_mse",
            "train_rel_l2",
            "val_mse",
            "val_mae",
            "val_rel_l2",
            "lr"
        ]
    )


# ============================================================
# Best model
# ============================================================

best_val = float(
    "inf"
)


best_model_path = os.path.join(

    OUT_DIR,

    "best_latent_neural_operator.pt"
)


# ============================================================
# Training
# ============================================================

epoch_bar = tqdm(

    range(
        1,
        EPOCHS + 1
    ),

    desc="Training",

    ncols=120
)


for epoch in epoch_bar:


    # ========================================================
    # Train
    # ========================================================

    train_mse, train_rel = train_one_epoch(

        model,

        train_loader,

        optimizer
    )


    # ========================================================
    # Validation
    # ========================================================

    (
        val_mse,
        val_mae,
        val_rel

    ) = evaluate(

        model,

        val_loader
    )


    # ========================================================
    # Scheduler
    # ========================================================

    scheduler.step()


    lr_now = optimizer.param_groups[
        0
    ]["lr"]


    # ========================================================
    # Write log
    # ========================================================

    with open(

        log_path,

        "a",

        newline=""

    ) as f:


        writer = csv.writer(
            f
        )


        writer.writerow(

            [
                epoch,
                train_mse,
                train_rel,
                val_mse,
                val_mae,
                val_rel,
                lr_now
            ]
        )


    # ========================================================
    # Save best model
    # ========================================================

    if val_rel < best_val:


        best_val = val_rel


        torch.save(

            model.state_dict(),

            best_model_path
        )


    # ========================================================
    # Print
    # ========================================================

    if (

        epoch == 1

        or

        epoch % 10 == 0
    ):


        print(

            f"\nEpoch {epoch:04d} | "

            f"Train MSE {train_mse:.4e} | "

            f"Train Rel {train_rel:.4e} | "

            f"Val MSE {val_mse:.4e} | "

            f"Val MAE {val_mae:.4e} | "

            f"Val Rel {val_rel:.4e}"
        )


    epoch_bar.set_postfix(

        train_mse=f"{train_mse:.2e}",

        val_mse=f"{val_mse:.2e}",

        rel=f"{val_rel:.2e}",

        lr=f"{lr_now:.1e}"
    )


# ============================================================
# Load best model
# ============================================================

print(
    "\nLoading best model..."
)


model.load_state_dict(

    torch.load(

        best_model_path,

        map_location=DEVICE,

        weights_only=True
    )
)


# ============================================================
# Final test
# ============================================================

(
    test_mse,
    test_mae,
    test_rel

) = evaluate(

    model,

    test_loader
)


print(
    "\nFinal Test Results"
)


print(
    f"Test MSE     : {test_mse:.6e}"
)


print(
    f"Test MAE     : {test_mae:.6e}"
)


print(
    f"Test Rel L2  : {test_rel:.6e}"
)


# ============================================================
# Save test results
# ============================================================

with open(

    os.path.join(
        OUT_DIR,
        "test_results.txt"
    ),

    "w"

) as f:


    f.write(
        "Final Test Results\n"
    )


    f.write(
        f"Test MSE     : "
        f"{test_mse:.6e}\n"
    )


    f.write(
        f"Test MAE     : "
        f"{test_mae:.6e}\n"
    )


    f.write(
        f"Test Rel L2  : "
        f"{test_rel:.6e}\n"
    )


# ============================================================
# Plot
# ============================================================

plot_loss(
    log_path
)


plot_prediction(

    model,

    test_dataset,

    sample_index=0
)


# ============================================================
# Finish
# ============================================================

print(
    f"\nBest validation Rel L2: "
    f"{best_val:.6e}"
)


print(
    f"Best model saved to: "
    f"{best_model_path}"
)


print(
    f"\nAll results saved to: "
    f"{OUT_DIR}"
)

Results will be saved to:
latent_neural_operator_results/run_20260826_151559
Using device: cuda
Total .mat samples found: 800

Dataset split:
Train samples: 640
Val samples  : 80
Test samples : 80

Calculating normalization statistics...


Statistics: 100%|█████████████████████████████████████████████████████████████████████| 640/640 [00:14<00:00, 43.60it/s]



Normalization statistics:
PT mean = 3.135567e+01
PT std  = 2.164740e+01
PA mean = -2.100887e+04
PA std  = 2.847824e+05

Batch check:
PT batch shape: torch.Size([4, 1, 501, 200])
PA batch shape: torch.Size([4, 1, 501, 200])

Checking model dimensions...
Input shape : torch.Size([1, 1, 501, 200])
Output shape: torch.Size([1, 1, 501, 200])

Trainable parameters: 39,271,681


Training:   0%|     | 1/1000 [01:09<19:20:36, 69.71s/it, lr=1.0e-03, rel=7.91e-01, train_mse=8.27e-01, val_mse=5.12e-01]


Epoch 0001 | Train MSE 8.2668e-01 | Train Rel 1.0466e+00 | Val MSE 5.1176e-01 | Val MAE 1.8864e-01 | Val Rel 7.9071e-01


...

Training:  24%|▏| 240/1000 [4:28:46<13:45:02, 65.13s/it, lr=8.6e-04, rel=3.08e-01, train_mse=2.84e-02, val_mse=1.22e-01]


Epoch 0240 | Train MSE 2.8361e-02 | Train Rel 1.7985e-01 | Val MSE 1.2226e-01 | Val MAE 6.6784e-02 | Val Rel 3.0804e-01


Training:  25%|▎| 250/1000 [4:39:43<13:37:17, 65.38s/it, lr=8.5e-04, rel=3.25e-01, train_mse=2.83e-02, val_mse=1.40e-01]


Epoch 0250 | Train MSE 2.8258e-02 | Train Rel 1.6876e-01 | Val MSE 1.3963e-01 | Val MAE 6.8205e-02 | Val Rel 3.2535e-01


Training:  25%|▎| 252/1000 [4:41:55<13:39:50, 65.76s/it, lr=8.5e-04, rel=2.99e-01, train_mse=2.76e-02, val_mse=1.14e-01]